In [1]:
# Imports
import pandas as pd
from random import randint
from src import *
from src.simulator import SIMULATOR

In [2]:
sim = SIMULATOR()

# --------------------------------------------
#               KERNEL CONFIGURATION
# --------------------------------------------
kernel_path = './kernels/mmul/'
kernel_number = 1 
column_usage = [True, False] 
nInstrPerCol = 48 
imem_add_start = 0 
srf_spm_addres = 0 
version="_bb16x16_1col_param"

sim.kernel_config(column_usage, nInstrPerCol, imem_add_start, srf_spm_addres, kernel_number)

In [3]:
# --------------------------------------------
#                DATA SIZES
# --------------------------------------------
# DISCO-CGRA Configuration
nRCs = 4
nElementsPerVWRSlice = 32
nColsCGRA = 2

# Basic Block
BB_ROWS_A = 8
BB_COLS_A = 16
BB_ROWS_B = 16
BB_COLS_B = 8
# C 16x8 to fit VWR
BB_ROWS_C = 16
BB_COLS_C = 8

In [4]:
# Our test
nRowsA = 16
nColsA = 9
nRowsB = nColsA
nColsB = 16
nRowsC = nRowsA
nColsC = nColsB

#matrix_A = np.random.randint(1, 15, size=(nRowsA*nColsA))
#matrix_B = np.random.randint(1, 15, size=(nColsA*nColsB))
matrix_A = np.array([i for i in range(nRowsA*nColsA)])
matrix_B = np.array([i for i in range(nRowsB*nColsB)])
matrix_C = np.zeros((nRowsA*nColsB), dtype=int)

In [5]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [6]:
# --------------------------------------------
#                LOAD SPM DATA
# --------------------------------------------
# SPM[0] = SRF
# SPM[1] = A00, SPM[2] = A10
# SPM[3] = B00, SPM[4] = B01
# SPM[5] = C00, SPM[6] = C10
# --------------------------------------------
# SRF[0] = nItLoop1 = n cols of A per block 
# SRF[1] = Line of the SPM where the block of A is stored
# SRF[2] = Line of the SPM where the block of B is stored
# SRF[3] = Line of the SPM where the block of C is stored
# SRF[4] = nItLoop2 = n cols of B per RC on the VWR
# SRF[5] = nItLoop3 = n rows of A per RC on the VWR
# SRF[6] = nItLoop4 = n VWRs for A
# SRF[7] = Not used
# --------------------------------------------

# Default SPM lines
srf_spm_line = 0
a_spm_line = 1
b_spm_line = 3
c_spm_line = 5


# Default SRF values
srf = [0 for i in range(N_ELEMS_PER_VWR)]
# Depend on the dimensions



# Col 0
srf[0]  = nColsA     # Loop1: Cols A per block
srf[1]  = 1      # SPM for A (same for both cols)
srf[2]  = 3      # SPM for B 
srf[3]  = 5      # SPM for C 
srf[4]  = 2      # Loop2: Cols B per RC 
srf[5]  = 2      # Loop3: Rows A per RC 
srf[6]  = 2      # Loop4: n VWRs for A 
srf[7]  = 0      # Not used
# Col 1
srf[8]  = nColsA     # Loop1: Cols A per block
srf[9]  = 1      # SPM for A (same for both cols)
srf[10] = 4      # SPM for B 
srf[11] = 6      # SPM for C 
srf[12] = 2      # Loop2: Cols B per RC 
srf[13] = 2      # Loop3: Rows A per RC 
srf[14] = 2      # Loop4: n VWRs for A 
srf[15] = 0      # Not used
sim.setSPMLine(srf_spm_line, srf.copy())

# Prepare A
nVWRforA = int(nRowsA/BB_ROWS_A)
if nRowsA%BB_ROWS_A != 0:
    nVWRforA+=1
nRowsAPerRC = 2
for nBlocksA in range(2):
    vector_A = [0 for i in range(N_ELEMS_PER_VWR)]
    # Write consecutively two consecutive rows of A per RC
    for rc in range(CGRA_ROWS):
        for idx_nRowsAPerRC in range(nRowsAPerRC):
            for cA in range(0, nColsA):
                idx_vA = rc*nElementsPerVWRSlice + idx_nRowsAPerRC*nColsA + cA
                rA = nBlocksA*BB_ROWS_A + rc*2 + idx_nRowsAPerRC 
                vector_A[idx_vA] = matrix_A[rA*nColsA + cA]
    sim.setSPMLine(a_spm_line, vector_A)
    print([int(x) for x in vector_A])
    a_spm_line+=1

# Prepare B
#matrix_B_reshaped = matrix_B.reshape(nColsA, nColsB) # Reshape into a 2D matrix
#matrix_B_transposed = matrix_B_reshaped.T.flatten() # Transpose and flatten back into a 1D array
nColsBPerRC = 2
for nBlocksB in range(2):
    vector_B = [0 for i in range(N_ELEMS_PER_VWR)]
    # Write consecutively two consecutive rows of A per RC
    for rc in range(CGRA_ROWS):
        for idx_nColsBPerRC in range(nColsBPerRC):
            for rB in range(0, nColsA):
                idx_vB = rc*nElementsPerVWRSlice + idx_nColsBPerRC*nColsA + rB
                cB = nBlocksB*BB_ROWS_A + rc*2 + idx_nColsBPerRC 
                vector_B[idx_vB] = matrix_B[rB*nColsB + cB]
    sim.setSPMLine(b_spm_line, vector_B)
    print([int(x) for x in vector_B])
    b_spm_line+=1

# Prepare C
# TODO: Parametrize
#aux_c_line = c_spm_line
#for iniCol in range(0, nColsC, BB_COLS_C):
#    aux_c = []
#    for row in range(BB_ROWS_C):
#        aux_c.extend(matrix_C[row*nColsC + iniCol:row*nColsC + iniCol + BB_COLS_C])
#    sim.setSPMLine(aux_c_line, aux_c)
#    aux_c_line+=1
printAsMatrix(matrix_A, nRowsA, nColsA)
printAsMatrix(matrix_B, nColsA, nColsB)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 16, 32, 48, 64, 80, 96, 112, 128, 1, 17, 33, 49,

In [7]:
sim.displaySPMLine(0)
sim.displaySPMLine(1)
sim.displaySPMLine(2)
sim.displaySPMLine(3)
sim.displaySPMLine(4)

SPM 0: [9, 1, 3, 5, 2, 2, 2, 0, 9, 1, 4, 6, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ]
SPM 1: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ]
SPM 2: [72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 90, 91, 92, 93, 94, 95, 96, 

In [8]:
# --------------------------------------------
#              COMPILE ASM TO HEX
# --------------------------------------------
sim.compileAsmToHex(kernel_path, kernel_number, version=version)

ASM to Hex
Processing file: ./kernels/mmul/instructions_asm_bb16x16_1col_param.csv...
Creating file: ./kernels/mmul/dsip_bitstream__bb16x16_1col_param.h
Creating file: ./kernels/mmul/instructions_hex_bb16x16_1col_param_autogen.csv


Finally, we load the kernel into the internal memory of the specialized units and run it.

In [9]:
# --------------------------------------------
#                 LOAD KERNEL
# --------------------------------------------

# This needs the hex instructions, if you don't provide them, generate then compiling the asm
sim.kernel_load(kernel_path, version=version + "_autogen", kernel_number=kernel_number)

# --------------------------------------------
#               SIMULATE EXECUTION
# --------------------------------------------
show_lcu = []
show_srf = []
show_lsu = []
show_rcs = [[],[],[],[]]
show_mxcu = []
display_ops = [show_lcu, show_lsu, show_mxcu, show_rcs, show_srf]

sim.run(kernel_number, display_ops=display_ops, max_iter=3000)

Processing file: ./kernels/mmul/instructions_hex_bb16x16_1col_param_autogen.csv...
---------------------
     PC[0]: 0
---------------------
LSU: NOP/LD.VWR SRF --> ALU res = 0
RC0: NOP --> ALU res = 0
RC1: NOP --> ALU res = 0
RC2: NOP --> ALU res = 0
RC3: NOP --> ALU res = 0
MXCU: NOP (VWR selected: 0, not writting SRF, R0: 0) --> ALU res = 0
LCU: NOP --> ALU res = 0
---------------------
     PC[0]: 1
---------------------
LSU: SADD R7, ZERO, SRF(3)/NOP --> ALU res = 5
RC0: NOP --> ALU res = 0
RC1: NOP --> ALU res = 0
RC2: NOP --> ALU res = 0
RC3: NOP --> ALU res = 0
MXCU: SADD R5, ZERO, LAST (VWR selected: 0, not writting SRF, R0: 0) --> ALU res = 31
LCU: NOP --> ALU res = 0
---------------------
     PC[0]: 2
---------------------
LSU: SADD R7, ZERO, SRF(2)/LD.VWR VWR_C --> ALU res = 3
RC0: NOP --> ALU res = 0
RC1: NOP --> ALU res = 0
RC2: NOP --> ALU res = 0
RC3: NOP --> ALU res = 0
MXCU: SADD R6, ZERO, LAST (VWR selected: 0, not writting SRF, R0: 0) --> ALU res = 31
LCU: NOP --> 

We can check it more rigorously. We can define our function in python and check that the output matches the CGRA output.

In [10]:
def mmul (in_A, in_B, nRowsA, nColsA, nColsB):
    out = np.zeros(nRowsA*nColsB)
    for i in range(nRowsA):
        for j in range(nColsB):
            sum = 0
            for k in range(nColsA):
                sum += int(in_A[i*nColsA + k] * in_B[k*nColsB + j])
            out[i*nColsB + j] = sum
    return [int(elem) for elem in out]

In [11]:
from itertools import groupby

def comprimir_rangos(arr):
    arr.sort()  # Asegurarse de que esté ordenado
    rangos = []
    
    for _, grupo in groupby(enumerate(arr), lambda x: x[1] - x[0]):
        grupo = [x[1] for x in grupo]  # Extraer los valores
        if len(grupo) > 1:
            rangos.append(f"{grupo[0]}-{grupo[-1]}")
        else:
            rangos.append(f"{grupo[0]}")

    return ", ".join(rangos)

def imprimir_por_linea(arr, tam_linea=8):
    for i in range(0, len(arr), tam_linea):
        print([int(x) for x in arr[i:i+tam_linea]])

In [12]:
def reordenarC(disco_cgra_res):
    nBloques = 16
    tamBloque = 8
    # Reorganizar los bloques en el orden correcto
    array_ordenado = []
    for i in range(nColsCGRA):
        ini = 2*tamBloque*i
        for j in range (nRCs):
            array_ordenado.extend(disco_cgra_res[ini:ini+2*tamBloque])
            ini += 4*tamBloque
    return array_ordenado

In [13]:
# Get output from the CGRA
disco_cgra_res_0 = sim.getSPMLine(c_spm_line)
disco_cgra_res_1 = sim.getSPMLine(c_spm_line + 1)
print(disco_cgra_res_0)
print(disco_cgra_res_1)

out_0 = reordenarC(disco_cgra_res_0)
out_1 = reordenarC(disco_cgra_res_1)

# Unir los bloques de C adecuadamente
disco_cgra_res = []
for i in range(0,N_ELEMS_PER_VWR, 8):
    disco_cgra_res.extend(out_0[i:i+8])
    disco_cgra_res.extend(out_1[i:i+8])

[3264, 53220, 54021, 54822, 52215, 3408, 56424, 53745, 3480, 58026, 55275, 8448, 8565, 8682, 8799, 0, 8916, 9033, 0, 9150, 9267, 0, 44736, 45420, 46104, 46788, 0, 47472, 48156, 0, 48840, 49524, 13632, 74118, 75243, 76368, 63069, 14424, 78618, 64923, 14820, 80868, 66777, 18816, 19095, 19374, 19653, 0, 19932, 20211, 0, 20490, 20769, 0, 55104, 55950, 56796, 57642, 0, 58488, 59334, 0, 60180, 61026, 24000, 95016, 96465, 97914, 73923, 25440, 100812, 76101, 26160, 103710, 78279, 29184, 29625, 30066, 30507, 0, 30948, 31389, 0, 31830, 32271, 0, 65472, 66480, 67488, 68496, 0, 69504, 70512, 0, 71520, 72528, 34368, 115914, 117687, 119460, 84777, 36456, 123006, 87279, 37500, 126552, 89781, 39552, 40155, 40758, 41361, 0, 41964, 42567, 0, 43170, 43773, 0, 75840, 77010, 78180, 79350, 0, 80520, 81690, 0, 82860, 84030]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [14]:
errors_idx = []
expected_output = mmul(matrix_A, matrix_B, nRowsA, nColsA, nColsB)
for i in range(len(expected_output)):
    if expected_output[i] != disco_cgra_res[i]:
        errors_idx.append(i)
if len(errors_idx) == 0:
    print("The result is correct!")
else:
    print("Oops, something went wrong. There are " + str(len(errors_idx)) + " errors.")
    print(comprimir_rangos(errors_idx))
    print("DISCO-CGRA result:")
    imprimir_por_linea(disco_cgra_res)
    print("Expected result:")
    imprimir_por_linea(expected_output)

Oops, something went wrong. There are 252 errors.
1-31, 33-63, 65-95, 97-255
DISCO-CGRA result:
[3264, 53220, 54021, 54822, 52215, 3408, 56424, 53745]
[0, 0, 0, 0, 0, 0, 0, 0]
[3480, 58026, 55275, 8448, 8565, 8682, 8799, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[13632, 74118, 75243, 76368, 63069, 14424, 78618, 64923]
[0, 0, 0, 0, 0, 0, 0, 0]
[14820, 80868, 66777, 18816, 19095, 19374, 19653, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[24000, 95016, 96465, 97914, 73923, 25440, 100812, 76101]
[0, 0, 0, 0, 0, 0, 0, 0]
[26160, 103710, 78279, 29184, 29625, 30066, 30507, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[34368, 115914, 117687, 119460, 84777, 36456, 123006, 87279]
[0, 0, 0, 0, 0, 0, 0, 0]
[37500, 126552, 89781, 39552, 40155, 40758, 41361, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[8916, 9033, 0, 9150, 9267, 0, 44736, 45420]
[0, 0, 0, 0, 0, 0, 0, 0]
[46104, 46788, 0, 47472, 48156, 0, 48840, 49524]
[0, 0, 0, 0, 0, 0, 0, 0]
[19932, 20211, 0, 20490, 20769, 0, 55104, 55950]
[0, 0, 0, 0, 0, 0, 0, 0]
[56796, 57642, 0, 58488, 59334, 0, 60180, 6102